In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
import logging
import os

from gsm_benchmarker.results_analyser.prompt_result import PromptResult
from gsm_benchmarker.results_analyser.plotting_utils import Colour

logger = logging.getLogger('notebook')

plt.style.use('default')
plt.style.use('seaborn-v0_8-muted')
plt.style.use('seaborn-v0_8-darkgrid')

In [ ]:
METRIC = "correct"
ALPHA = 0.05

In [ ]:
OUTPUTS = Path("results_pres/outputs").resolve()
os.makedirs(OUTPUTS, exist_ok=True)
OUTPUTS_FOLDER = str(OUTPUTS) + "/"

In [ ]:
significant_models = [
    'phi-2',
    'Phi-3.5-mini-instruct',
    'gemma-2b',
    'gemma-2-9b',
    'Mathstral-7B-v0.1',
    'Meta-Llama-3-8B-Instruct',
    'gemma-7b-it',
    'Meta-Llama-3-8B',
    'Mistral-7B-Instruct-v0.1',
    'gemma-2-2b'
]

In [ ]:
result_kwargs = dict(metric=METRIC, save_dest=OUTPUTS)
pp = Path("../../../data/gsm-symbolic/outputs").resolve()


gsm_result = PromptResult(
    pp / "noq_default_full__12_05/final",
    colour=Colour('green'),
    full_label="GSM prompt",
    **result_kwargs
)


short_code_result = PromptResult(
    pp / "noq_code_short__12_05/final",
    colour=Colour("mediumpurple").lighten(0.2),
    full_label="Simple code-output prompt",
    short_label="code-simple",
    baseline=gsm_result.mres,
    **result_kwargs
)

short_code_result_clean = short_code_result.get_clean_data_object()

long_code_result = PromptResult(
    pp / "noq_code_long__12_05/final",
    colour=Colour("rebeccapurple"),
    full_label="Structured code-output prompt",
    short_label="code-structured",
    baseline=gsm_result.mres,
    **result_kwargs
)

long_code_result_clean = long_code_result.get_clean_data_object()


In [ ]:
vek = {'model_order': significant_models}

In [ ]:
def match_index(s, df):
    if not isinstance(df.index, pd.MultiIndex) or df.index.nlevels < 2:
        return s

    if df.index.nlevels > 2:
        raise RuntimeError('Matching dataframe index for a multi-index of more than 2 levels not implemented')

    index_names = df.index.names
    s_matched = pd.DataFrame({
        k: s for k in df.index.get_level_values(index_names[1]).unique()
    }).stack()
    s_matched.index.names = index_names
    return s_matched


def print_comparison_summary(comparison_df):
    print(f"\nAgreement rate: {comparison_df.agreement.sum()}/{len(comparison_df)}")

    if (~comparison_df.agreement).sum():
        print(f"Disagreement cases:\n{comparison_df[~comparison_df.agreement][
            ['original_p', 'clean_p', 'original_failed', 'clean_failed']]}")

    er = comparison_df.exclusion_rate
    print(f"\nExclusion rate max / mean: {er.max():.3f} / {er.mean():.3f}\n")


def compare(orig_res, clean_res, alpha=0.05, effect='variant', model_order: list[str] | None = None):
    (orig_df, orig_sum), (clean_df, clean_sum) = [getattr(res, f'{effect}_effect') for res in (orig_res, clean_res)]
    comparison_df = pd.DataFrame({
        'original_p': orig_df.p_value,
        'clean_p': clean_df.p_value
    })
    comparison_df['original_significant'] = comparison_df['original_p'] < alpha
    comparison_df['clean_significant'] = comparison_df['clean_p'] < alpha
    comparison_df['agreement'] = comparison_df['original_significant'] == comparison_df['clean_significant']

    comparison_df['estimate_diff'] = clean_df['estimate'] - orig_df['estimate']

    n_resp, n_clean_resp = [res.mres.variants['main'].full_data.groupby('model').size() for res in (orig_res, clean_res)]
    er = 1 - n_clean_resp / n_resp
    comparison_df['exclusion_rate'] = match_index(er, comparison_df)

    if model_order is not None:
        comparison_df.sort_index(
            level='model',
            key=lambda idx: idx.map({model: i for i, model in enumerate(model_order)}),
            inplace=True
        )

    fi_orig, fi_clean = [
        (
                sum_df.fit_failed | sum_df.is_singular | sum_df.convergence_messages
        ) for sum_df in (orig_sum, clean_sum)
    ]

    comparison_df['original_failed'] = match_index(fi_orig, comparison_df)
    comparison_df['clean_failed'] = match_index(fi_clean, comparison_df)

    print_comparison_summary(comparison_df)

    return comparison_df


In [ ]:
compare(short_code_result, short_code_result_clean, **vek)


In [ ]:
compare(long_code_result, long_code_result_clean, **vek)

In [ ]:
compare(short_code_result, short_code_result_clean, effect='number', **vek)

In [ ]:
compare(long_code_result, long_code_result_clean, effect='number', **vek)
